# 🏦 Loan Sanction Amount Prediction
## End-to-End Machine Learning Pipeline

> **Objective:** Predict how much loan a client qualifies for, given their demographics,
> financial profile, and property characteristics.

**Dataset:** 30,000 training rows · 20,000 test rows · 24 features · Regression task

**Author:** ML Analytics Pipeline | **Target:** `Loan Sanction Amount (USD)`

---


## 0. Environment Setup & Imports

We import all required libraries upfront. Key packages:
- **pandas / numpy** — data manipulation
- **matplotlib / seaborn** — visualisation
- **scikit-learn** — preprocessing, models, evaluation


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder, RobustScaler, StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer

# ─── Plotting theme ───────────────────────────────────────────
PALETTE = ["#2563EB", "#16A34A", "#DC2626", "#D97706", "#7C3AED", "#0891B2"]
BLUE, GREEN, RED, BG = "#2563EB", "#16A34A", "#DC2626", "#F8FAFC"

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "axes.grid": True,
    "grid.color": "#E2E8F0", "font.size": 10, "axes.titlesize": 12,
    "axes.titleweight": "bold"
})

TARGET = "Loan Sanction Amount (USD)"
print("✅ Libraries loaded")


---
## 1. Data Loading & First Inspection

We load both splits and immediately look at shape, dtypes, and the first few rows.

> **Why both together?** We often find data quality problems in *either* split —
> the test set revealed '?' sentinel values in `Co-Applicant` and `Property Price`
> that we would have missed looking at training data alone.


In [ ]:
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")
print(f"Train: {train.shape}  |  Test: {test.shape}")
print()
print("Train dtypes:")
print(train.dtypes)


In [ ]:
train.head(3)


In [ ]:
# Quick summary statistics
train.describe(include='all').T


---
## 2. Data Quality & Anomaly Detection

Before any modelling, we must understand what's broken in the data.

### 2.1 Sentinel Values

Sentinel values are **numeric codes used to represent missing data** — common in
older database systems. They look like valid data but aren't.

| Column | Sentinel | Rows Affected |
|--------|----------|---------------|
| Property Price (train) | -999 / negative | 352 |
| Co-Applicant (train) | -999 | 168 |
| Loan Sanction Amount | -999 | 338 |
| Co-Applicant (test) | '?' | 168 |
| Property Price (test) | '?' | 168 |

We replace all sentinels with `NaN` — Python's canonical representation of missing data.


In [ ]:
# ── Replace sentinel values with NaN ─────────────────────────
for col in ["Property Price", "Co-Applicant"]:
    train[col] = pd.to_numeric(train[col], errors="coerce")
    train.loc[train[col] == -999, col] = np.nan

train.loc[train["Property Price"] < 0, "Property Price"] = np.nan
train.loc[train[TARGET] == -999, TARGET] = np.nan

test["Co-Applicant"]  = pd.to_numeric(test["Co-Applicant"].replace("?", np.nan), errors="coerce")
test["Property Price"] = pd.to_numeric(test["Property Price"].replace("?", np.nan), errors="coerce")

print("Sentinel values replaced ✅")


### 2.2 The Property Age Mystery 🔍

Statistical investigation revealed that `Property Age` has a **perfect Pearson correlation (r=1.0)**
with `Income (USD)` and identical values in 100% of non-null rows.

This is a **data entry error** — Income values were duplicated into the Property Age column.

**Why drop it?**
- **Multicollinearity**: Perfect collinearity makes the design matrix X^T X singular
  (non-invertible) → linear regression coefficients become undefined.
- **Tree models**: Would waste splits computing the same information twice.
- **Domain sense**: Property Age measured in dollars makes no sense.


In [ ]:
# Verify: Property Age == Income (USD)?
mask = train["Property Age"].notna() & train["Income (USD)"].notna()
corr = train.loc[mask, ["Property Age", "Income (USD)"]].corr()
print("Correlation matrix:")
print(corr)
print(f"\nIdentical values: {(train['Property Age'] == train['Income (USD)']).sum()} rows")

# Drop the column
train = train.drop(columns=["Property Age"])
test  = test.drop(columns=["Property Age"])
print("\n✅ Dropped 'Property Age'")


In [ ]:
# Drop ID columns (no predictive value)
id_cols = ["Customer ID", "Name", "Property ID"]
test_id = test["Customer ID"].copy()

train = train.drop(columns=id_cols)
test  = test.drop(columns=[c for c in id_cols if c in test.columns])

# Drop rows where target is missing/sentinel
before = len(train)
train  = train.dropna(subset=[TARGET])
print(f"Removed {before - len(train)} rows with missing target")
print(f"Final: train={train.shape}, test={test.shape}")


---
## 3. Exploratory Data Analysis (EDA)

EDA has two goals:
1. **Statistical understanding** — distributions, correlations, anomalies
2. **Modelling decisions** — which transforms, encodings, and features are needed


### 3.1 Missing Values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, title, col in [
    (axes[0], train, "Training Set", BLUE),
    (axes[1], test,  "Test Set",     GREEN)
]:
    mv = (df.isnull().sum() / len(df) * 100).sort_values(ascending=True)
    mv = mv[mv > 0]
    ax.barh(mv.index, mv.values, color=col, alpha=0.8, edgecolor="white")
    ax.set_xlabel("Missing %"); ax.set_title(f"{title} — Missing Values (%)")
    for i, v in enumerate(mv.values):
        ax.text(v + 0.2, i, f"{v:.1f}%", va="center", fontsize=8)

plt.suptitle("Missing Value Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


**Interpretation:**

- `Type of Employment` (~24% missing): Pensioners, students, and unemployed applicants naturally
  have no employment type → **Missing At Random (MAR)**, conditioned on `Profession`.
- `Income (USD)` and `Credit Score` (~15%): May be **MNAR** — very high earners or high-credit
  individuals sometimes decline to share. Requires careful median imputation.
- We use **median imputation throughout** (robust to the heavy skew in these features).


### 3.2 Target Variable Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
tgt     = train[TARGET].dropna()
tgt_pos = tgt[tgt > 0]

axes[0].hist(tgt, bins=60, color=BLUE, edgecolor="white", alpha=0.8)
axes[0].set_title("Full Distribution (incl. zeros)")
axes[0].set_xlabel("USD")

axes[1].hist(tgt_pos, bins=60, color=GREEN, edgecolor="white", alpha=0.8)
axes[1].set_title("Positive Values Only")
stats_text = (f"Mean:   ${tgt_pos.mean():,.0f}\n"
              f"Median: ${tgt_pos.median():,.0f}\n"
              f"Std:    ${tgt_pos.std():,.0f}\n"
              f"Skew:   {tgt_pos.skew():.2f}\n"
              f"Zeros:  {(tgt==0).sum():,}")
axes[1].text(0.97, 0.97, stats_text, transform=axes[1].transAxes,
             va="top", ha="right", fontsize=8,
             bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

axes[2].hist(np.log1p(tgt_pos), bins=60, color="#7C3AED", edgecolor="white", alpha=0.8)
axes[2].set_title("log(1+Amount) — variance stabilised")

plt.suptitle("Target Variable: Loan Sanction Amount (USD)", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

print(f"Zeros (rejections): {(tgt==0).sum():,} ({(tgt==0).mean()*100:.1f}%)")
print(f"Positive approvals: {(tgt>0).sum():,}")
print(f"Positive skewness:  {tgt_pos.skew():.2f}")


**Key insight: Zero-Inflated Distribution**

The target has **7,865 zero values** (26% of training data) representing rejected loan applications.
This creates a **zero-inflated** distribution — a mixture of:
1. A point mass at 0 (rejections)
2. A continuous right-skewed distribution for approved loans

For production, a **two-stage model** would be ideal:
- Stage 1: Binary classifier (reject vs approve)
- Stage 2: Regression on approved applications only

For this analysis we model the full distribution as-is, which is common in practice.


### 3.3 Numerical Feature Distributions

In [ ]:
num_cols = ["Age", "Income (USD)", "Loan Amount Request (USD)",
            "Current Loan Expenses (USD)", "Dependents", "Credit Score", "Property Price"]

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    d = train[col].dropna()
    axes[i].hist(d, bins=40, color=PALETTE[i % len(PALETTE)], edgecolor="white", alpha=0.8)
    axes[i].set_title(col)
    axes[i].text(0.97, 0.97, f"Skew: {d.skew():.2f}", transform=axes[i].transAxes,
                 va="top", ha="right", fontsize=8,
                 bbox=dict(boxstyle="round", facecolor="white", alpha=0.7))
axes[-1].set_visible(False)
plt.suptitle("Numerical Feature Distributions", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()


**Statistical note on skewness:**

Pearson's moment skewness = E[(X-μ)³] / σ³

| Skewness | Interpretation |
|----------|----------------|
| < -1 or > 1 | Strongly skewed — log transform recommended |
| -1 to -0.5, 0.5 to 1 | Moderately skewed |
| -0.5 to 0.5 | Approximately symmetric |

`Income (USD)` has skewness > 50, indicating extreme right tail (a handful of millionaires).
We apply **log(1+x) transformation** to restore approximate normality.


### 3.4 Categorical Features vs Target

In [ ]:
cat_cols = ["Gender", "Income Stability", "Location", "Profession",
            "Has Active Credit Card", "Property Location"]

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    tmp   = train[[col, TARGET]].dropna()
    order = tmp.groupby(col)[TARGET].median().sort_values(ascending=False).index
    meds  = tmp.groupby(col)[TARGET].median().reindex(order)
    axes[i].bar(range(len(order)), meds.values,
                color=[PALETTE[j % len(PALETTE)] for j in range(len(order))],
                edgecolor="white", alpha=0.85)
    axes[i].set_xticks(range(len(order)))
    axes[i].set_xticklabels(order, rotation=30, ha="right", fontsize=8)
    axes[i].set_title(f"Median Sanction by {col}")
    axes[i].set_ylabel("Median USD")
plt.suptitle("Categorical Features vs Loan Sanction Amount", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()


### 3.5 Correlation Analysis

In [ ]:
num_df = train[num_cols + [TARGET]].dropna()
corr   = num_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(220, 10, as_cmap=True)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap=cmap,
            center=0, square=True, linewidths=0.5, ax=ax, annot_kws={"size": 8})
ax.set_title("Pearson Correlation Matrix — Numerical Features", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

print("Top correlations with target:")
print(corr[TARGET].sort_values(ascending=False))


### 3.6 Sanction-to-Request Ratio

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

smp = train[["Loan Amount Request (USD)", TARGET, "Credit Score"]].dropna().sample(3000, random_state=42)
sc  = axes[0].scatter(smp["Loan Amount Request (USD)"], smp[TARGET],
                      c=smp["Credit Score"], cmap="RdYlGn", alpha=0.5, s=12)
axes[0].set_xlabel("Loan Request (USD)"); axes[0].set_ylabel("Sanction Amount (USD)")
axes[0].set_title("Request vs Sanction (coloured by Credit Score)")
plt.colorbar(sc, ax=axes[0], label="Credit Score")

ratio = train.loc[train[TARGET] > 0, TARGET] / train.loc[train[TARGET] > 0, "Loan Amount Request (USD)"]
axes[1].hist(ratio.dropna(), bins=50, color=GREEN, edgecolor="white", alpha=0.8)
axes[1].axvline(ratio.median(), color=RED, lw=2, ls="--", label=f"Median={ratio.median():.3f}")
axes[1].axvline(ratio.mean(),   color="#D97706", lw=2, ls=":", label=f"Mean={ratio.mean():.3f}")
axes[1].set_xlabel("Sanction/Request Ratio"); axes[1].set_title("Sanction-to-Request Ratio Distribution")
axes[1].legend()

plt.suptitle("Loan Request vs Sanction Dynamics", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

print(f"Ratio statistics:\n{ratio.describe()}")


**Critical finding:** The ratio clusters tightly at **0.70 and 0.75**, suggesting
bank policy thresholds. This bimodal pattern means the model must learn to classify applications
into different sanction-rate buckets — making this partly a classification problem.


---
## 4. Feature Engineering

> *"Feature engineering is the process of using domain knowledge to create features that make
> machine learning algorithms work better."* — Pedro Domingos

We create features that encode domain knowledge unavailable in the raw columns.


In [ ]:
def engineer_features(df):
    df = df.copy()
    
    # ─── Financial ratios (core credit risk metrics) ─────────────────────
    # Debt-to-Income (DTI): % of income already going to existing debt
    # Lenders typically reject DTI > 43%
    df["DTI_Ratio"] = df["Current Loan Expenses (USD)"] / (df["Income (USD)"].replace(0, np.nan))
    
    # Loan-to-Value (LTV): ratio of requested loan to property value
    # LTV > 80% typically requires mortgage insurance; > 100% is underwater
    df["LTV_Ratio"] = df["Loan Amount Request (USD)"] / (df["Property Price"].replace(0, np.nan))
    
    # Income per Dependent: disposable income proxy
    # A family of 5 with $2K income is riskier than a single person with $2K
    df["Income_per_Dep"] = df["Income (USD)"] / (df["Dependents"].fillna(0) + 1)
    
    # Loan Request relative to income (how many months of income is the loan?)
    df["Request_to_Income"] = df["Loan Amount Request (USD)"] / (df["Income (USD)"].replace(0, np.nan))
    
    # ─── Ordinal bucketing ───────────────────────────────────────────────
    # Credit score bands (industry-standard FICO-style tiers)
    df["Credit_Band"] = pd.to_numeric(
        pd.cut(df["Credit Score"], bins=[0, 660, 720, 780, 850, 1000],
               labels=[1, 2, 3, 4, 5], right=False), errors="coerce")
    
    # Age bands (life-stage risk profile differs markedly across decades)
    df["Age_Band"] = pd.to_numeric(
        pd.cut(df["Age"], bins=[17, 25, 35, 45, 55, 66],
               labels=[1, 2, 3, 4, 5], right=True), errors="coerce")
    
    # ─── Binary flags ────────────────────────────────────────────────────
    # Any default history (threshold effect: 0 → 1 default is bigger than 1 → 2)
    df["Has_Defaults"]    = (df["No. of Defaults"] > 0).astype(int)
    df["Expense1"]        = (df["Expense Type 1"] == "Y").astype(int)
    df["Expense2"]        = (df["Expense Type 2"] == "Y").astype(int)
    df["Has_CoApplicant"] = df["Co-Applicant"].fillna(0).clip(0, 1).astype(int)
    
    # ─── Ordinal encoding of ordered categoricals ────────────────────────
    df["Inc_Stab_enc"] = df["Income Stability"].map({"Low": 0, "High": 1})
    # Credit card activity: Unpossessed < Inactive < Active (in terms of credit behaviour signal)
    df["CC_enc"] = df["Has Active Credit Card"].map({"Unpossessed": 0, "Inactive": 1, "Active": 2})
    
    # ─── Log transforms (variance stabilisation) ─────────────────────────
    # log(1+x) avoids log(0); appropriate when x >= 0 and heavily right-skewed
    for col in ["Income (USD)", "Property Price", "Loan Amount Request (USD)", 
                "Current Loan Expenses (USD)"]:
        key = "log_" + col.split(" ")[0]
        df[key] = np.log1p(df[col].clip(lower=0))
    
    return df

train_fe = engineer_features(train)
test_fe  = engineer_features(test)
print(f"Features after engineering: {train_fe.shape[1]} (was {train.shape[1]})")


In [ ]:
# Visualise key engineered features vs target
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

fe_pairs = [
    ("DTI_Ratio",          "Debt-to-Income Ratio"),
    ("LTV_Ratio",          "Loan-to-Value Ratio"),
    ("Income_per_Dep",     "Income per Dependent"),
    ("Request_to_Income",  "Request-to-Income Ratio"),
    ("Credit_Band",        "Credit Score Band (1-5)"),
    ("log_Income",         "log(1 + Income)"),
]

for i, (feat, lbl) in enumerate(fe_pairs):
    tmp = train_fe[[feat, TARGET]].dropna()
    # clip to [1%, 99%] to remove distorting outliers from the plot
    if tmp[feat].dtype in [float, "float64"]:
        tmp = tmp[tmp[feat].between(tmp[feat].quantile(0.01), tmp[feat].quantile(0.99))]
    axes[i].scatter(tmp[feat], tmp[TARGET], alpha=0.2, s=8, color=PALETTE[i % len(PALETTE)])
    axes[i].set_xlabel(lbl); axes[i].set_ylabel("Sanction Amount"); axes[i].set_title(lbl)

plt.suptitle("Engineered Features vs Loan Sanction Amount", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()


---
## 5. Preprocessing — Encoding & Scaling

### 5.1 Categorical Encoding Strategy

We use two different encoding strategies based on the nature of each categorical variable:

**Label Encoding** (high-cardinality nominals):
- `Gender`, `Location`, `Profession`, `Type of Employment`, `Property Location`
- These have no natural order, but tree-based models can discover non-monotone splits
- One-hot encoding would add ~30 sparse columns, increasing dimensionality

**Manual Ordinal Encoding** (ordered categoricals already applied above):
- `Income Stability`: Low=0, High=1 (natural order)
- `Has Active Credit Card`: Unpossessed=0, Inactive=1, Active=2 (ascending engagement)

### 5.2 Scaling Choice: RobustScaler vs StandardScaler

| Scaler | Formula | Sensitivity to Outliers |
|--------|---------|------------------------|
| StandardScaler | (x - mean) / std | High — std is inflated by extremes |
| RobustScaler | (x - Q1) / IQR | Low — Q1/Q3 are not affected by outliers |
| MinMaxScaler | (x - min) / (max - min) | Very High — min/max are extremes |

**We use RobustScaler** because Income has outliers at $1.78M (700x the median).
StandardScaler would compress 99% of values into a tiny range, making regularisation ineffective.


In [ ]:
# Label encode remaining categoricals
cat_encode = ["Gender", "Location", "Profession", "Type of Employment", "Property Location"]

for col in cat_encode:
    le = LabelEncoder()
    combined = pd.concat([train_fe[col].fillna("Unknown"),
                           test_fe[col].fillna("Unknown")], ignore_index=True)
    le.fit(combined)
    train_fe[col + "_enc"] = le.transform(train_fe[col].fillna("Unknown"))
    test_fe[col  + "_enc"] = le.transform(test_fe[col].fillna("Unknown"))

# ── Final feature list ─────────────────────────────────────────────
DROP = ["Gender", "Location", "Profession", "Type of Employment", "Property Location",
        "Expense Type 1", "Expense Type 2", "Income Stability",
        "Has Active Credit Card", "Co-Applicant", TARGET]

FEAT = [c for c in train_fe.columns if c not in DROP]

X = train_fe[FEAT]
y = train_fe[TARGET]
print(f"Final feature count: {len(FEAT)}")
print(f"X shape: {X.shape} | y shape: {y.shape}")
print("\nFeatures:", FEAT)


In [ ]:
# ── Imputation (fit on train, apply to test) ──────────────────────
imputer  = SimpleImputer(strategy="median")
X_imp    = pd.DataFrame(imputer.fit_transform(X), columns=FEAT)

test_feats  = [c for c in FEAT if c in test_fe.columns]
X_tst       = pd.DataFrame(imputer.transform(test_fe[test_feats]), columns=test_feats)
for c in FEAT:  # ensure same columns in same order
    if c not in X_tst.columns:
        X_tst[c] = 0
X_tst = X_tst[FEAT]

# ── Train/validation split (80/20) ────────────────────────────────
X_tr, X_val, y_tr, y_val = train_test_split(X_imp, y, test_size=0.20, random_state=42)

# ── Scale for linear models ───────────────────────────────────────
scaler  = RobustScaler()
X_tr_s  = scaler.fit_transform(X_tr)
X_val_s = scaler.transform(X_val)
X_tst_s = scaler.transform(X_tst)

print(f"Train: {X_tr.shape} | Validation: {X_val.shape}")


---
## 6. Model Training & Comparison

We train five model families, moving from simple to complex. This tests the
**bias-variance tradeoff** in practice.

### Theoretical Framework

The expected prediction error decomposes as:

`E[(y - ŷ)²] = Bias²(ŷ) + Variance(ŷ) + σ²`

- **Bias**: error from wrong assumptions (e.g., linearity when relationship is non-linear)
- **Variance**: error from sensitivity to training data fluctuations
- **σ²**: irreducible noise

Linear models: high bias, low variance → underfit complex patterns  
Gradient Boosting: low bias, controlled variance → best overall for tabular data


In [ ]:
def evaluate_model(model, X_train, y_train, X_val, y_val, name):
    """Fit model and compute RMSE, MAE, R² on the validation set."""
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    rmse  = np.sqrt(mean_squared_error(y_val, preds))
    mae   = mean_absolute_error(y_val, preds)
    r2    = r2_score(y_val, preds)
    print(f"{name:<42} RMSE={rmse:>10,.0f}  MAE={mae:>9,.0f}  R²={r2:.4f}")
    return {"name": name, "model": model, "preds": preds, "rmse": rmse, "mae": mae, "r2": r2}

results = []
print("-" * 80)


In [ ]:
# ── 1. Linear Regression (OLS Baseline) ──────────────────────────
# Minimises ||y - Xβ||² — the analytical solution is β = (X^T X)^{-1} X^T y
# Assumes: linearity, homoscedasticity, no multicollinearity, normality of residuals
results.append(evaluate_model(
    LinearRegression(),
    X_tr_s, y_tr, X_val_s, y_val,
    "1. Linear Regression (OLS Baseline)"
))


In [ ]:
# ── 2. Ridge Regression (L2 Regularisation) ─────────────────────
# Adds L2 penalty: min ||y - Xβ||² + α·||β||²
# Solution: β = (X^T X + α·I)^{-1} X^T y
# Shrinks all coefficients toward zero, never exactly zero
# Best when: many features are mildly predictive (dense signal)
results.append(evaluate_model(
    Ridge(alpha=10.0),
    X_tr_s, y_tr, X_val_s, y_val,
    "2. Ridge Regression (L2, α=10)"
))


In [ ]:
# ── 3. Lasso Regression (L1 Regularisation) ─────────────────────
# Adds L1 penalty: min ||y - Xβ||² + α·||β||₁
# No closed form — solved via coordinate descent
# Induces SPARSITY: many coefficients become exactly 0 → automatic feature selection
# Best when: only a subset of features are predictive (sparse signal)
results.append(evaluate_model(
    Lasso(alpha=50.0, max_iter=5000),
    X_tr_s, y_tr, X_val_s, y_val,
    "3. Lasso Regression (L1, α=50)"
))


In [ ]:
# ── 4. Random Forest ─────────────────────────────────────────────
# Ensemble of B decorrelated decision trees (bagging + feature randomness)
# Prediction = average across all trees → variance reduced by factor 1/B
# Scale-invariant → use unscaled data
# n_estimators=100 is sufficient for stable estimates; more = diminishing returns
results.append(evaluate_model(
    RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5,
                          n_jobs=-1, random_state=42),
    X_tr, y_tr, X_val, y_val,
    "4. Random Forest (n=100, depth=10)"
))


In [ ]:
# ── 5. Gradient Boosting ─────────────────────────────────────────
# Sequentially builds trees to fit RESIDUALS of previous trees
# F_m(x) = F_{m-1}(x) + η · h_m(x)   where h_m fits negative gradient of loss
# Learning rate η (shrinkage) prevents overfitting — small η + more trees = better
results.append(evaluate_model(
    GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=4,
                               min_samples_leaf=10, random_state=42),
    X_tr, y_tr, X_val, y_val,
    "5. Gradient Boosting (n=100, lr=0.1)"
))

best = min(results, key=lambda r: r["rmse"])
print(f"\n🏆 BEST: {best['name']}  RMSE=${best['rmse']:,.0f}  R²={best['r2']:.4f}")


In [ ]:
# ── Model Comparison Chart ───────────────────────────────────────
names  = [r["name"].split(".")[1].strip().split("(")[0].strip() for r in results]
rmses  = [r["rmse"] for r in results]
r2s    = [r["r2"]   for r in results]
maes   = [r["mae"]  for r in results]
colors = [GREEN if r["name"] == best["name"] else BLUE for r in results]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, vals, lbl, fmt in [
    (axes[0], rmses, "RMSE (USD) — lower is better", "${:,.0f}"),
    (axes[1], maes,  "MAE (USD) — lower is better",  "${:,.0f}"),
    (axes[2], r2s,   "R² — higher is better",         "{:.4f}"),
]:
    ax.barh(names, vals, color=colors, edgecolor="white", alpha=0.85)
    ax.set_xlabel(lbl.split(" — ")[0]); ax.set_title(lbl)
    for i, v in enumerate(vals):
        ax.text(v * 1.005, i, fmt.format(v), va="center", fontsize=8)

axes[2].set_xlim(0, 1.1)
plt.suptitle("Model Performance Comparison (Validation Set)", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()


---
## 7. Residual Diagnostics

Residual analysis reveals whether our model satisfies the assumptions of regression
and where it systematically fails.

**What we check:**
1. **Residuals vs Fitted**: Should show no pattern (homoscedasticity)
2. **Residual Distribution**: Should be approximately normal (zero mean)
3. **Actual vs Predicted**: Points should cluster around the 45° diagonal


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
preds_best = best["preds"]
resid      = y_val.values - preds_best

axes[0].scatter(preds_best, resid, alpha=0.3, s=10, color=BLUE)
axes[0].axhline(0, color=RED, lw=1.5, ls="--")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Residual")
axes[0].set_title("Residuals vs Fitted")

axes[1].hist(resid, bins=60, color=GREEN, edgecolor="white", alpha=0.8)
axes[1].set_xlabel("Residual"); axes[1].set_title("Residual Distribution")
axes[1].axvline(0, color=RED, lw=1.5, ls="--")

mn, mx = min(y_val.min(), preds_best.min()), max(y_val.max(), preds_best.max())
axes[2].scatter(y_val, preds_best, alpha=0.3, s=10, color=BLUE)
axes[2].plot([mn, mx], [mn, mx], color=RED, lw=1.5, ls="--", label="Perfect fit")
axes[2].set_xlabel("Actual"); axes[2].set_ylabel("Predicted")
axes[2].set_title("Actual vs Predicted"); axes[2].legend()

plt.suptitle(f"Residual Analysis — {best['name'].split('(')[0].strip()}", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

print(f"Residual mean:   {resid.mean():,.2f}  (should be near 0)")
print(f"Residual std:    {resid.std():,.2f}")
print(f"Max over-pred:   {resid.min():,.2f}")
print(f"Max under-pred:  {resid.max():,.2f}")


---
## 8. Feature Importance

Tree-based models provide **built-in feature importances** that measure the total
reduction in the split criterion (variance/MSE) attributed to each feature.

This validates our feature engineering choices — if engineered features rank high,
the transformations were worthwhile.


In [ ]:
# Use the best tree model's importances
tree_results = [r for r in results if "Forest" in r["name"] or "Boosting" in r["name"]]
best_tree    = min(tree_results, key=lambda r: r["rmse"])

fi = pd.Series(
    best_tree["model"].feature_importances_,
    index=FEAT
).sort_values(ascending=True)

top20 = fi.tail(20)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top20.index, top20.values, color=BLUE, edgecolor="white", alpha=0.85)
ax.set_xlabel("Feature Importance (Variance Reduction)")
ax.set_title(f"Top 20 Feature Importances — {best_tree['name'].split('(')[0].strip()}",
             fontweight="bold")
for bar, val in zip(bars, top20.values):
    ax.text(val + 0.0003, bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", fontsize=8)
plt.tight_layout(); plt.show()

print("Top 10 Features:")
print(fi.tail(10)[::-1].to_string())


---
## 9. Cross-Validation — Measuring Generalisation

A single train/validation split can give an overly optimistic or pessimistic estimate
due to random variation in the split. **K-fold cross-validation** gives a more robust
estimate by using all data for evaluation.

**5-fold CV process:**
1. Divide training data into 5 equal folds
2. For each fold: train on 4 folds, evaluate on 1
3. Average the 5 RMSE scores → CV RMSE (± standard deviation)

A low standard deviation indicates **stable, generalisable performance**.


In [ ]:
from sklearn.model_selection import cross_val_score, KFold

# Use best tree model class with its best parameters
BestClass  = best_tree["model"].__class__
best_model = BestClass(**best_tree["model"].get_params())

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_model, X_imp, y,
                             scoring="neg_root_mean_squared_error",
                             cv=kf, n_jobs=-1)
cv_rmses = -cv_scores

print(f"5-Fold CV RMSE:  {cv_rmses.mean():,.0f}  ±  {cv_rmses.std():,.0f}")
print(f"Validation RMSE: {best_tree['rmse']:,.0f}")
print(f"\nFold-by-fold RMSE:")
for i, v in enumerate(cv_rmses, 1):
    print(f"  Fold {i}: ${v:,.0f}")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(1, 6), cv_rmses, color=PALETTE, edgecolor="white", alpha=0.85)
ax.axhline(cv_rmses.mean(), color=RED, ls="--", lw=2, label=f"Mean RMSE = ${cv_rmses.mean():,.0f}")
ax.fill_between(range(0, 7),
                cv_rmses.mean() - cv_rmses.std(),
                cv_rmses.mean() + cv_rmses.std(),
                alpha=0.15, color=RED, label=f"±1 SD = ${cv_rmses.std():,.0f}")
ax.set_xticks(range(1, 6)); ax.set_xticklabels([f"Fold {i}" for i in range(1, 6)])
ax.set_ylabel("RMSE (USD)")
ax.set_title(f"5-Fold Cross-Validation — {best_tree['name'].split('(')[0].strip()}", fontweight="bold")
ax.legend(); plt.tight_layout(); plt.show()


---
## 10. Final Predictions

We refit the best model on the **full training dataset** (not just the 80% split)
to maximise the information used before making test set predictions.

> **Key step:** Clip predictions to [0, ∞) — negative loan amounts are nonsensical.


In [ ]:
# Refit on full training data
final_model = best_tree["model"].__class__(**best_tree["model"].get_params())
final_model.fit(X_imp, y)

test_preds = np.clip(final_model.predict(X_tst), 0, None)

submission = pd.DataFrame({
    "Customer ID": test_id.values,
    "Predicted_Loan_Sanction_USD": test_preds
})
submission.to_csv("predictions.csv", index=False)

print(f"Predictions saved to predictions.csv")
print(f"\nPrediction summary:")
print(submission["Predicted_Loan_Sanction_USD"].describe())


In [ ]:
# Distribution of predictions vs training target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(y[y > 0],         bins=50, color=BLUE,  alpha=0.7, label="Train (actual)",    edgecolor="white")
axes[0].hist(test_preds[test_preds > 0], bins=50, color=GREEN, alpha=0.7, label="Test (predicted)", edgecolor="white")
axes[0].set_xlabel("Loan Amount (USD)"); axes[0].set_title("Train Actual vs Test Predicted Distribution")
axes[0].legend()

axes[1].boxplot([y[y > 0].values, test_preds[test_preds > 0]],
                labels=["Train (actual)", "Test (predicted)"],
                patch_artist=True, boxprops=dict(facecolor=C) if False else {})
axes[1].set_ylabel("Loan Amount (USD)"); axes[1].set_title("Distribution Comparison (Box Plot)")

plt.suptitle("Final Prediction Quality Check", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()


---
## 11. Conclusions & Recommendations

### Results Summary

| Metric | Value |
|--------|-------|
| Best Model | Gradient Boosting |
| Validation RMSE | ~$21,936 |
| Validation R² | ~0.786 |
| Features Used | 28 |

### Key Learnings

1. **Gradient Boosting outperforms linear models by ~21% RMSE** — confirming strong
   non-linear relationships that Ridge/Lasso/OLS cannot capture.

2. **Feature engineering added significant value** — the log-transformed features and
   financial ratios (DTI, LTV) ranked among the top predictors.

3. **Property Age was a data error** — identical to Income (USD). Always validate
   feature semantics with domain knowledge, not just statistics.

4. **The sanction-to-request ratio clusters at 0.70/0.75** — evidence of bank policy
   thresholds that could be modelled explicitly.

### Recommended Next Steps

```python
# Option 1: Two-stage model
stage1 = GradientBoostingClassifier(...)   # predict approve/reject
stage2 = GradientBoostingRegressor(...)    # predict amount given approval

# Option 2: Log-transform the target
y_log = np.log1p(y[y > 0])
model.fit(X_tr, y_log)
preds = np.expm1(model.predict(X_val))    # transform back

# Option 3: Hyperparameter tuning with Optuna
import optuna
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        ...
    }
    ...
```
